<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="left">Run Inference with Your YOLO Model</h1>
<h4 align="left">SUBSIM Playday on EDITO · Written by the KSO Team</h4>

Now that you have a trained model, let's see what it can actually detect.

Your model was trained to recognise three species found in Swedish coastal waters:

| Class | Species | Notes |
|:------|:--------|:------|
| 0 | *Ctenolabrus rupestris* (goldsinny wrasse) | Native |
| 1 | *Gobius niger* (black goby) | Native |
| 2 | **Neogobius melanostomus** (round goby) | **Invasive** |

The round goby is one of Europe's most damaging invasive fish species, competing with native gobies for food and habitat. Early detection matters - and that's exactly what your model can do.

In this notebook you'll run your model on **unseen images** containing round goby and see whether it can spot them.

1. **Configure** – Point to your model weights and the held-out images
2. **Predict on images** – See detections overlaid on each frame
3. **Predict on video** *(optional)* – Process a short clip and save the annotated output

Run cells **top to bottom**.

---
## Phase 1: Configuration

Point to your trained model and the images you want to run inference on.

> **Which model?** Use the `best.pt` from the training notebook, or try the pre-trained golden model from the playday data.

In [ ]:
from pathlib import Path

# ── Model ──
model_path = ""          # <-- Path to your best.pt weights
                         #     e.g. "models/my_first_model/weights/best.pt"  (from training notebook)
                         #     or   "SUBSIM_playday_data/03_golden_model/best.pt"  (pre-trained)

# ── Inputs ──
images_dir = ""          # <-- Folder of images to run inference on
                         #     e.g. "SUBSIM_playday_data/02_inference_inputs/images"

# ── Settings ──
conf_thres = 0.5         # Minimum confidence to show a detection

# ── Validate ──
model_path = Path(model_path)
images_dir = Path(images_dir)

assert model_path.exists(), f"Model not found: {model_path}"
assert images_dir.exists(), f"Images folder not found: {images_dir}"

images = sorted([p for p in images_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
print(f"Model:   {model_path}")
print(f"Images:  {len(images)} found in {images_dir}")

---
## Phase 2: Detect Round Goby in New Images

Runs the model on images it has **never seen during training** and displays the results with bounding box overlays.

Can it find the invasive round goby? Look for the class labels on each detection box.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO(str(model_path))
results = model.predict(images, conf=conf_thres, verbose=False)

# Display results in a grid
n = len(results)
cols = min(n, 3)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 7 * rows))
if n == 1:
    axes = [axes]
else:
    axes = axes.flat if hasattr(axes, "flat") else [axes]

for i, (ax, r) in enumerate(zip(axes, results)):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    n_det = len(r.boxes)
    ax.set_title(f"{Path(r.path).name}  ({n_det} detection{'s' if n_det != 1 else ''})", fontsize=12)
    ax.axis("off")

# Hide unused axes
for j in range(i + 1, len(list(axes))):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

# Print summary
total = sum(len(r.boxes) for r in results)
print(f"\nTotal: {total} detections across {n} images")

---
## Phase 3: Predict on Video *(Optional)*

Processes the first N seconds of a video clip and saves an annotated version with bounding boxes drawn on each frame.

> **Skip this cell** if you don't have a video file, or set `DO_VIDEO = False`.

In [ ]:
DO_VIDEO = False  # <-- Set True to enable

# ── Video settings ──
video_path = ""          # <-- e.g. "SUBSIM_playday_data/02_inference_inputs/video/short_clip_01.mp4"
clip_seconds = 10        # How many seconds to process

# ─────────────────────

if not DO_VIDEO:
    print("Video inference: disabled (set DO_VIDEO = True to enable)")
else:
    import cv2

    video_path = Path(video_path)
    assert video_path.exists(), f"Video not found: {video_path}"

    # Read video properties
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    max_frames = int(fps * clip_seconds)
    cap.release()

    out_path = Path("inference_output") / f"annotated_{video_path.stem}.mp4"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Processing {clip_seconds}s of {video_path.name} ({w}x{h} @ {fps:.0f} fps)...")

    model = YOLO(str(model_path))
    stream = model.predict(source=str(video_path), conf=conf_thres, stream=True, verbose=False)

    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    frame_count = 0
    for r in stream:
        if frame_count >= max_frames:
            break
        writer.write(r.plot())
        frame_count += 1
    writer.release()

    print(f"\nSaved {frame_count} annotated frames to:")
    print(f"  {out_path}")

---
## ✅ Done!

You've run your trained model on new data and seen its detections.

### Try It Yourself

- **Compare models:** Go back to Phase 1 and swap `model_path` to the golden model or a different training run
- **Adjust confidence:** Lower `conf_thres` to 0.3 to see more (noisier) detections, or raise it to 0.7 for only high-confidence ones

---

### Saving Your Work

All files in this Jupyter instance will be deleted when you shut down the SUBSIM service. To keep your results:

```python
!mc cp inference_output/ s3/$S3_BUCKET/inference_output/ --recursive
```

See the [EDITO File Explorer](https://datalab.dive.edito.eu/file-explorer) to verify your saved files.